# Retail Sales Analytics — End-to-End Data Analysis**Author:** Yasar Khan Sattar Khan PathanThis notebook walks through a complete data analysis workflow on a retail sales dataset:1. Data Loading & Inspection2. Data Cleaning3. Exploratory Data Analysis (EDA)4. Business Insights & KPIs5. Export of a clean, analysis-ready dataset for Power BI

## 1. Import Libraries & Load Data

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snssns.set_style('whitegrid')plt.rcParams['figure.figsize'] = (10, 5)df = pd.read_csv('../data/retail_sales_raw.csv')print(df.shape)df.head()

## 2. Initial Inspection

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## 3. Data CleaningSteps taken:- Remove exact duplicate rows- Standardize `Region` text casing- Fill missing `Discount` values with 0 (assume no discount applied)- Fill missing `Profit` using category-level average profit margin- Convert date columns to datetime

In [ ]:
df_clean = df.drop_duplicates().copy()df_clean['Region'] = df_clean['Region'].str.title()df_clean['Discount'] = df_clean['Discount'].fillna(0)avg_margin = (df_clean.groupby('Category').apply(lambda x: (x['Profit']/x['Sales']).mean()))missing_profit_mask = df_clean['Profit'].isnull()df_clean.loc[missing_profit_mask, 'Profit'] = df_clean.loc[missing_profit_mask, 'Category'].map(avg_margin) * df_clean.loc[missing_profit_mask, 'Sales']df_clean['OrderDate'] = pd.to_datetime(df_clean['OrderDate'])df_clean['ShipDate'] = pd.to_datetime(df_clean['ShipDate'])df_clean['ShippingDays'] = (df_clean['ShipDate'] - df_clean['OrderDate']).dt.daysdf_clean['OrderMonth'] = df_clean['OrderDate'].dt.to_period('M').astype(str)df_clean['ProfitMargin'] = (df_clean['Profit'] / df_clean['Sales']).round(3)print('Remaining nulls:')print(df_clean.isnull().sum().sum())df_clean.head()

## 4. Key Business Metrics (KPIs)

In [ ]:
total_sales = df_clean['Sales'].sum()total_profit = df_clean['Profit'].sum()total_orders = df_clean['OrderID'].nunique()avg_order_value = total_sales / total_ordersprofit_margin_pct = (total_profit / total_sales) * 100print(f'Total Sales: ₹{total_sales:,.0f}')print(f'Total Profit: ₹{total_profit:,.0f}')print(f'Total Orders: {total_orders:,}')print(f'Average Order Value: ₹{avg_order_value:,.0f}')print(f'Overall Profit Margin: {profit_margin_pct:.1f}%')

## 5. Sales & Profit by Category

In [ ]:
cat_summary = df_clean.groupby('Category')[['Sales','Profit']].sum().sort_values('Sales', ascending=False)cat_summary['ProfitMargin%'] = (cat_summary['Profit']/cat_summary['Sales']*100).round(1)cat_summary

In [ ]:
fig, ax = plt.subplots()cat_summary['Sales'].plot(kind='bar', ax=ax, color='#1F3864')ax.set_title('Total Sales by Category')ax.set_ylabel('Sales (₹)')plt.xticks(rotation=30)plt.tight_layout()plt.savefig('../images/sales_by_category.png', dpi=120)plt.show()

## 6. Monthly Sales Trend

In [ ]:
monthly = df_clean.groupby('OrderMonth')['Sales'].sum().reset_index()fig, ax = plt.subplots(figsize=(12,5))ax.plot(monthly['OrderMonth'], monthly['Sales'], marker='o', color='#1F3864')ax.set_title('Monthly Sales Trend (2024–2025)')ax.set_ylabel('Sales (₹)')plt.xticks(rotation=60)plt.tight_layout()plt.savefig('../images/monthly_sales_trend.png', dpi=120)plt.show()

## 7. Regional Performance

In [ ]:
region_summary = df_clean.groupby('Region')[['Sales','Profit']].sum().sort_values('Sales', ascending=False)fig, ax = plt.subplots()region_summary['Sales'].plot(kind='bar', ax=ax, color='#2E5395')ax.set_title('Sales by Region')plt.xticks(rotation=0)plt.tight_layout()plt.savefig('../images/sales_by_region.png', dpi=120)plt.show()

## 8. Customer Segment Analysis

In [ ]:
seg_summary = df_clean.groupby('Segment')[['Sales','Profit']].sum()fig, ax = plt.subplots()seg_summary['Sales'].plot(kind='pie', autopct='%1.1f%%', ax=ax, colors=['#1F3864','#4472C4','#8EA9DB'])ax.set_ylabel('')ax.set_title('Sales Share by Customer Segment')plt.tight_layout()plt.savefig('../images/sales_by_segment.png', dpi=120)plt.show()

## 9. Discount Impact on Profit Margin

In [ ]:
fig, ax = plt.subplots()sns.scatterplot(data=df_clean, x='Discount', y='ProfitMargin', hue='Category', alpha=0.5, ax=ax)ax.set_title('Discount vs. Profit Margin')ax.axhline(0, color='red', linestyle='--', linewidth=1)plt.tight_layout()plt.savefig('../images/discount_vs_margin.png', dpi=120)plt.show()

## 10. Key Insights- **Electronics and Furniture** drive the highest revenue but have thinner profit margins than Clothing and Beauty.- **Discounts above 15–20%** push several orders into negative or near-zero profit margin — a candidate for a discount policy review.- Sales show **seasonal peaks** around certain months, useful for inventory and staffing planning.- The **Consumer segment** contributes the largest share of sales, suggesting marketing spend is well-aligned but corporate upsell may be underexplored.- **Regional performance** is fairly balanced, with West and North slightly ahead — worth investigating what's driving the gap with East/Central.

## 11. Export Clean Dataset for Power BI

In [ ]:
df_clean.to_csv('../data/retail_sales_clean.csv', index=False)print('Exported clean dataset to data/retail_sales_clean.csv')print(df_clean.shape)